# Spam Detection Model
This notebook explore email spam detection using the Enron dataset.

## 1. Imports and Setup

In [ ]:
import pandas as pd
import re
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


## 2. Load and Clean Data

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\W+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def load_and_preprocess(file_path):
    df = pd.read_csv(file_path)
    df["Message"] = df["Message"].fillna("")
    df["Subject"] = df["Subject"].fillna("")
    df["text"] = df["Subject"] + " " + df["Message"]
    df["text"] = df["text"].apply(clean_text)
    df["label"] = df["Spam/Ham"].map({"ham": 0, "spam": 1})
    df = df.dropna(subset=["label"]).copy()
    df["label"] = df["label"].astype(int)
    return df

df = load_and_preprocess("data/enron_spam_data.csv")
df.head()

## 3. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)
print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

## 4. Model Pipeline

In [ ]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, stop_words="english")),
    ("nb", MultinomialNB())
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))